# Part 11: Feature Store

[← Back to Index](Index.ipynb)

**Quick Reference Guide for Feature Store Architecture**

---
## 11.1 Feature Store Fundamentals

**What is a Feature Store?**

A centralized repository for storing, managing, and serving ML features for both training and inference.

**Why Feature Stores?**
- **Reusability:** Share features across teams/models
- **Consistency:** Same features for training and serving
- **Efficiency:** Precomputed features, avoid redundant computation
- **Governance:** Track lineage, versions, metadata
- **Performance:** Optimized storage and serving

**Key Components:**
1. **Feature Registry:** Metadata, schemas, lineage
2. **Offline Store:** Historical features for training
3. **Online Store:** Low-latency features for serving
4. **Feature Pipeline:** Transform raw data to features

---
## 11.2 Offline Feature Store

**Purpose:** Store historical features for model training

**Characteristics:**
- Batch processing
- Large datasets
- Point-in-time correctness
- Optimized for throughput

**Technologies:** S3, Data Lake, BigQuery, Snowflake, Delta Lake

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Simulated Feature Store Implementation

class OfflineFeatureStore:
    """Simple offline feature store using pandas"""
    
    def __init__(self):
        self.features = {}
        self.metadata = {}
    
    def register_feature_group(self, name, description, features, entity_key):
        """Register a feature group"""
        self.metadata[name] = {
            'description': description,
            'features': features,
            'entity_key': entity_key,
            'created_at': datetime.now()
        }
        print(f"Feature group '{name}' registered successfully")
    
    def write_features(self, feature_group_name, data):
        """Write features to offline store"""
        if feature_group_name not in self.metadata:
            raise ValueError(f"Feature group '{feature_group_name}' not registered")
        
        if feature_group_name not in self.features:
            self.features[feature_group_name] = data
        else:
            self.features[feature_group_name] = pd.concat(
                [self.features[feature_group_name], data], ignore_index=True
            )
        
        print(f"Written {len(data)} rows to '{feature_group_name}'")
    
    def get_historical_features(self, feature_group_name, entity_ids=None, 
                               start_date=None, end_date=None):
        """Retrieve historical features"""
        if feature_group_name not in self.features:
            raise ValueError(f"No data for feature group '{feature_group_name}'")
        
        data = self.features[feature_group_name].copy()
        
        # Filter by entity IDs
        if entity_ids:
            entity_key = self.metadata[feature_group_name]['entity_key']
            data = data[data[entity_key].isin(entity_ids)]
        
        # Filter by date range
        if 'timestamp' in data.columns:
            if start_date:
                data = data[data['timestamp'] >= start_date]
            if end_date:
                data = data[data['timestamp'] <= end_date]
        
        return data
    
    def list_feature_groups(self):
        """List all feature groups"""
        return pd.DataFrame([
            {
                'name': name,
                'description': meta['description'],
                'num_features': len(meta['features']),
                'entity_key': meta['entity_key']
            }
            for name, meta in self.metadata.items()
        ])

# Example usage
offline_store = OfflineFeatureStore()

# Register feature group
offline_store.register_feature_group(
    name='user_features',
    description='User demographic and behavior features',
    features=['age', 'total_purchases', 'avg_purchase_value', 'days_since_signup'],
    entity_key='user_id'
)

# Generate sample data
np.random.seed(42)
n_users = 1000
dates = pd.date_range('2023-01-01', periods=30, freq='D')

data_list = []
for user_id in range(1, n_users + 1):
    for date in dates:
        data_list.append({
            'user_id': user_id,
            'timestamp': date,
            'age': np.random.randint(18, 70),
            'total_purchases': np.random.randint(0, 100),
            'avg_purchase_value': np.random.uniform(10, 500),
            'days_since_signup': np.random.randint(1, 1000)
        })

user_features = pd.DataFrame(data_list)

# Write features
offline_store.write_features('user_features', user_features)

# List feature groups
print("\nFeature Groups:")
print(offline_store.list_feature_groups())

# Retrieve features
historical_features = offline_store.get_historical_features(
    'user_features',
    entity_ids=[1, 2, 3],
    start_date='2023-01-15',
    end_date='2023-01-20'
)

print(f"\nRetrieved {len(historical_features)} rows")
print(historical_features.head())

---
## 11.3 Online Feature Store

**Purpose:** Serve features in real-time for model inference

**Characteristics:**
- Low latency (< 10ms)
- High throughput
- Latest feature values
- Key-value lookup

**Technologies:** Redis, DynamoDB, Cassandra, Bigtable

In [ ]:
class OnlineFeatureStore:
    """Simple online feature store using dictionary (Redis-like)"""
    
    def __init__(self):
        self.store = {}  # {feature_group: {entity_id: {feature: value}}}
    
    def materialize_features(self, feature_group_name, data, entity_key):
        """Materialize features from offline to online store"""
        if feature_group_name not in self.store:
            self.store[feature_group_name] = {}
        
        # Get latest values for each entity
        if 'timestamp' in data.columns:
            data = data.sort_values('timestamp').groupby(entity_key).last()
        
        # Store features
        for entity_id, row in data.iterrows():
            feature_dict = row.drop('timestamp', errors='ignore').to_dict()
            self.store[feature_group_name][entity_id] = feature_dict
        
        print(f"Materialized {len(data)} entities to online store")
    
    def get_online_features(self, feature_group_name, entity_ids, features=None):
        """Get features for online serving (low latency)"""
        if feature_group_name not in self.store:
            raise ValueError(f"Feature group '{feature_group_name}' not in online store")
        
        results = []
        for entity_id in entity_ids:
            if entity_id in self.store[feature_group_name]:
                feature_dict = self.store[feature_group_name][entity_id].copy()
                
                if features:
                    feature_dict = {k: v for k, v in feature_dict.items() if k in features}
                
                results.append(feature_dict)
            else:
                results.append(None)
        
        return results
    
    def update_feature(self, feature_group_name, entity_id, feature_name, value):
        """Update a single feature value (real-time update)"""
        if feature_group_name not in self.store:
            self.store[feature_group_name] = {}
        
        if entity_id not in self.store[feature_group_name]:
            self.store[feature_group_name][entity_id] = {}
        
        self.store[feature_group_name][entity_id][feature_name] = value

# Example usage
online_store = OnlineFeatureStore()

# Materialize from offline to online
latest_features = user_features.copy()
latest_features = latest_features.set_index('user_id')
online_store.materialize_features('user_features', latest_features, 'user_id')

# Get features for online serving (fast lookup)
import time

start = time.time()
features = online_store.get_online_features(
    'user_features',
    entity_ids=[1, 2, 3],
    features=['age', 'total_purchases', 'avg_purchase_value']
)
latency = (time.time() - start) * 1000

print(f"\nOnline lookup latency: {latency:.2f}ms")
print("\nFeatures retrieved:")
for i, f in enumerate(features, 1):
    print(f"User {i}: {f}")

# Real-time update
online_store.update_feature('user_features', 1, 'total_purchases', 150)
print("\nAfter real-time update:")
print(online_store.get_online_features('user_features', [1]))

---
## 11.4 Feature Store Architecture

**Complete Architecture Components:**

### Feature Registry

**Purpose:** Central catalog of features with metadata

In [ ]:
class FeatureRegistry:
    """Feature metadata and governance"""
    
    def __init__(self):
        self.registry = {}
    
    def register_feature(self, feature_name, metadata):
        """Register feature with metadata"""
        self.registry[feature_name] = {
            **metadata,
            'registered_at': datetime.now(),
            'version': metadata.get('version', '1.0'),
        }
        print(f"Feature '{feature_name}' registered (v{self.registry[feature_name]['version']})")
    
    def get_feature_metadata(self, feature_name):
        """Get feature metadata"""
        return self.registry.get(feature_name)
    
    def search_features(self, **filters):
        """Search features by criteria"""
        results = []
        for name, meta in self.registry.items():
            match = all(meta.get(k) == v for k, v in filters.items())
            if match:
                results.append({'name': name, **meta})
        return pd.DataFrame(results)

# Example
registry = FeatureRegistry()

registry.register_feature(
    'user_age',
    {
        'description': 'User age in years',
        'type': 'int',
        'source': 'user_table',
        'owner': 'data-team',
        'tags': ['demographic', 'user'],
        'version': '1.0'
    }
)

registry.register_feature(
    'total_purchases',
    {
        'description': 'Total number of purchases by user',
        'type': 'int',
        'source': 'orders_table',
        'owner': 'data-team',
        'tags': ['behavior', 'user'],
        'version': '1.0'
    }
)

# Search features
print("\nFeatures owned by 'data-team':")
print(registry.search_features(owner='data-team'))

### Feature Versioning

**Why:** Track feature changes, enable rollback, reproducibility

In [ ]:
class FeatureVersionControl:
    """Version control for features"""
    
    def __init__(self):
        self.versions = {}  # {feature_name: {version: data}}
    
    def save_version(self, feature_name, version, data, metadata=None):
        """Save a feature version"""
        if feature_name not in self.versions:
            self.versions[feature_name] = {}
        
        self.versions[feature_name][version] = {
            'data': data,
            'metadata': metadata or {},
            'timestamp': datetime.now()
        }
        print(f"Saved {feature_name} version {version}")
    
    def get_version(self, feature_name, version):
        """Retrieve specific feature version"""
        if feature_name not in self.versions:
            raise ValueError(f"Feature '{feature_name}' not found")
        
        if version not in self.versions[feature_name]:
            raise ValueError(f"Version '{version}' not found")
        
        return self.versions[feature_name][version]['data']
    
    def list_versions(self, feature_name):
        """List all versions of a feature"""
        if feature_name not in self.versions:
            return []
        
        versions = []
        for ver, info in self.versions[feature_name].items():
            versions.append({
                'version': ver,
                'timestamp': info['timestamp'],
                'metadata': info['metadata']
            })
        return pd.DataFrame(versions)

# Example
version_control = FeatureVersionControl()

# Save v1
version_control.save_version(
    'user_features',
    'v1.0',
    user_features.head(100),
    {'description': 'Initial version', 'author': 'data-team'}
)

# Save v2 (with improvements)
version_control.save_version(
    'user_features',
    'v2.0',
    user_features.head(200),
    {'description': 'Added more users', 'author': 'data-team'}
)

# List versions
print("\nFeature versions:")
print(version_control.list_versions('user_features'))

# Get specific version
v1_data = version_control.get_version('user_features', 'v1.0')
print(f"\nv1.0 has {len(v1_data)} rows")

### Feature Pipeline

**Purpose:** Transform raw data into features

In [ ]:
class FeaturePipeline:
    """Feature transformation pipeline"""
    
    def __init__(self, name):
        self.name = name
        self.transformations = []
    
    def add_transformation(self, func, name):
        """Add transformation step"""
        self.transformations.append({'func': func, 'name': name})
    
    def execute(self, data):
        """Execute pipeline"""
        result = data.copy()
        
        for step in self.transformations:
            print(f"Executing: {step['name']}")
            result = step['func'](result)
        
        return result

# Example pipeline
pipeline = FeaturePipeline('user_feature_pipeline')

# Add transformations
pipeline.add_transformation(
    lambda df: df.assign(age_group=pd.cut(df['age'], bins=[0, 25, 40, 60, 100], 
                                          labels=['18-25', '26-40', '41-60', '60+'])),
    'Create age groups'
)

pipeline.add_transformation(
    lambda df: df.assign(purchase_frequency=df['total_purchases'] / 
                        (df['days_since_signup'] + 1)),
    'Calculate purchase frequency'
)

pipeline.add_transformation(
    lambda df: df.assign(high_value_customer=(df['avg_purchase_value'] > 200).astype(int)),
    'Flag high-value customers'
)

# Execute pipeline
print("\nExecuting feature pipeline:")
transformed_features = pipeline.execute(user_features.head())
print("\nTransformed features:")
print(transformed_features[['user_id', 'age', 'age_group', 'purchase_frequency', 
                            'high_value_customer']].head())

---
### Popular Feature Store Solutions

### Feast (Open Source)

```python
# Install: pip install feast
from feast import FeatureStore, Entity, Feature, FeatureView, FileSource
from feast.value_type import ValueType

# Define entity
user = Entity(name="user", value_type=ValueType.INT64, description="User ID")

# Define data source
user_source = FileSource(
    path="data/user_features.parquet",
    event_timestamp_column="timestamp",
)

# Define feature view
user_features_view = FeatureView(
    name="user_features",
    entities=["user"],
    ttl=timedelta(days=7),
    features=[
        Feature(name="age", dtype=ValueType.INT32),
        Feature(name="total_purchases", dtype=ValueType.INT32),
        Feature(name="avg_purchase_value", dtype=ValueType.FLOAT),
    ],
    source=user_source,
)

# Initialize feature store
store = FeatureStore(repo_path=".")

# Get online features
features = store.get_online_features(
    features=["user_features:age", "user_features:total_purchases"],
    entity_rows=[{"user": 1}, {"user": 2}]
).to_dict()
```

---
### Quick Reference Guide

**Architecture Pattern:**

```
Raw Data → Feature Pipeline → Offline Store (training)
                           ↓
                    Online Store (serving)
                           ↓
                    Feature Registry (metadata)
```

**Offline vs Online:**

| Aspect | Offline Store | Online Store |
|--------|--------------|-------------|
| **Purpose** | Training | Inference |
| **Latency** | Seconds-Minutes | < 10ms |
| **Data Volume** | Large (TB-PB) | Small (GB) |
| **Storage** | S3, Data Lake | Redis, DynamoDB |
| **Access Pattern** | Batch | Key-value lookup |
| **Time Range** | Historical | Latest only |

**Best Practices:**
1. **Consistency:** Ensure training-serving skew is minimal
2. **Monitoring:** Track feature quality, staleness, drift
3. **Governance:** Document ownership, lineage, SLAs
4. **Testing:** Validate feature transformations
5. **Versioning:** Enable reproducibility and rollback
6. **Security:** Control access, encrypt sensitive features
7. **Cost:** Optimize storage and compute costs
8. **Point-in-time:** Prevent data leakage in offline store

**Popular Solutions:**
- **Feast:** Open-source, Kubernetes-native
- **Tecton:** Enterprise feature platform
- **AWS SageMaker Feature Store:** AWS-native
- **Google Vertex AI Feature Store:** GCP-native
- **Databricks Feature Store:** Unified with Delta Lake
- **Hopsworks:** Open-source platform

---
[← Back to Index](Index.ipynb)